<a href="https://colab.research.google.com/github/Suhasnangineni/House-Price-Prediction/blob/main/AerosheildAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##22/07/26 SUHAS N

Package Installation

In [ ]:
!pip install -q pyarrow
import pandas as pd, numpy as np, sklearn
print(pd.__version__, np.__version__, sklearn.__version__)

2.2.2 2.0.2 1.6.1


Drive mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Upload CMAPSSData.zip to a shared Drive folder once, then point here:
DATA_ZIP = '/content/drive/MyDrive/aerosheild/CMAPSSData.zip'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Unzip & Confirm

In [ ]:
import zipfile, os

RAW_DIR = '/content/cmapss_raw'
os.makedirs(RAW_DIR, exist_ok=True)

with zipfile.ZipFile(DATA_ZIP, 'r') as z:
    z.extractall(RAW_DIR)

print(os.listdir(RAW_DIR))

['RUL_FD004.txt', 'train_FD001.txt', 'Damage Propagation Modeling.pdf', 'train_FD004.txt', 'RUL_FD001.txt', 'test_FD003.txt', 'RUL_FD002.txt', 'train_FD003.txt', 'test_FD002.txt', 'test_FD004.txt', 'train_FD002.txt', 'readme.txt', 'test_FD001.txt', 'RUL_FD003.txt']


Preparing CMAPSS

In [ ]:
"""
AeroShield AI - CMAPSS Dataset Preparation Pipeline
=====================================================
Turns the raw NASA CMAPSS turbofan engine files into a clean,
labeled, normalized, and windowed dataset ready for model training.

Usage:
    python prepare_cmapss.py --subset FD001 --data_dir ./CMAPSS_raw --out_dir ./processed

Input files expected in data_dir (from the official NASA zip):
    train_FD00X.txt, test_FD00X.txt, RUL_FD00X.txt
"""

import argparse
import os

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

COLUMN_NAMES = (
    ["unit", "cycle", "op_setting_1", "op_setting_2", "op_setting_3"]
    + [f"sensor_{i}" for i in range(1, 22)]
)

RUL_CAP = 125          # standard piecewise-linear RUL ceiling used across CMAPSS literature
SEQ_LEN = 30            # sliding window length for sequence models (LSTM/Transformer)
VAL_SPLIT = 0.2         # fraction of training engines held out for validation
RANDOM_STATE = 42

# Subsets with multiple operating regimes (need regime clustering before scaling)
MULTI_REGIME_SUBSETS = {"FD002", "FD004"}
N_REGIMES = 6


# ---------------------------------------------------------------------------
# Step 1-2: Load raw files and assign column names
# ---------------------------------------------------------------------------

def load_raw(data_dir: str, subset: str):
    train_path = os.path.join(data_dir, f"train_{subset}.txt")
    test_path = os.path.join(data_dir, f"test_{subset}.txt")
    rul_path = os.path.join(data_dir, f"RUL_{subset}.txt")

    train = pd.read_csv(train_path, sep=r"\s+", header=None, names=COLUMN_NAMES)
    test = pd.read_csv(test_path, sep=r"\s+", header=None, names=COLUMN_NAMES)
    rul_test = pd.read_csv(rul_path, sep=r"\s+", header=None, names=["RUL"])

    print(f"[load] {subset}: train={train.shape}, test={test.shape}, "
          f"engines(train)={train['unit'].nunique()}, engines(test)={test['unit'].nunique()}")
    return train, test, rul_test


# ---------------------------------------------------------------------------
# Step 3: Compute RUL label for every row of the training set
# ---------------------------------------------------------------------------

def add_train_rul(train: pd.DataFrame) -> pd.DataFrame:
    max_cycle = train.groupby("unit")["cycle"].max().reset_index()
    max_cycle.columns = ["unit", "max_cycle"]

    train = train.merge(max_cycle, on="unit")
    train["RUL"] = train["max_cycle"] - train["cycle"]
    train = train.drop(columns="max_cycle")
    return train


# ---------------------------------------------------------------------------
# Step 4: Cap RUL (piecewise-linear degradation assumption)
# ---------------------------------------------------------------------------

def cap_rul(df: pd.DataFrame, cap: int = RUL_CAP) -> pd.DataFrame:
    df["RUL"] = df["RUL"].clip(upper=cap)
    return df


# ---------------------------------------------------------------------------
# Step 5: Drop sensors that carry no information (constant across dataset)
# ---------------------------------------------------------------------------

def drop_constant_sensors(train: pd.DataFrame, test: pd.DataFrame):
    sensor_cols = [c for c in train.columns if c.startswith("sensor_")]
    constant_sensors = [c for c in sensor_cols if train[c].nunique() <= 1]

    print(f"[filter] dropping constant sensors: {constant_sensors or 'none'}")
    train = train.drop(columns=constant_sensors)
    test = test.drop(columns=constant_sensors)
    return train, test, constant_sensors


# ---------------------------------------------------------------------------
# Step 6: Cluster operating regimes (only needed for multi-condition subsets)
# ---------------------------------------------------------------------------

def add_regime_clusters(train: pd.DataFrame, test: pd.DataFrame, subset: str):
    if subset not in MULTI_REGIME_SUBSETS:
        train["regime"] = 0
        test["regime"] = 0
        return train, test, None

    op_cols = ["op_setting_1", "op_setting_2", "op_setting_3"]
    km = KMeans(n_clusters=N_REGIMES, random_state=RANDOM_STATE, n_init=10)
    train["regime"] = km.fit_predict(train[op_cols])
    test["regime"] = km.predict(test[op_cols])

    print(f"[regime] {subset}: clustered into {N_REGIMES} operating regimes")
    return train, test, km


# ---------------------------------------------------------------------------
# Step 7: Normalize sensor readings (per-regime if applicable)
# ---------------------------------------------------------------------------

def normalize_sensors(train: pd.DataFrame, test: pd.DataFrame, subset: str):
    sensor_cols = [c for c in train.columns if c.startswith("sensor_")]
    train[sensor_cols] = train[sensor_cols].astype(float)
    test[sensor_cols] = test[sensor_cols].astype(float)

    if subset in MULTI_REGIME_SUBSETS:
        # Fit a separate scaler per regime so readings are comparable within
        # the same operating condition, not distorted by cross-regime scale.
        scalers = {}
        for regime in sorted(train["regime"].unique()):
            scaler = MinMaxScaler()
            train_mask = train["regime"] == regime
            train.loc[train_mask, sensor_cols] = scaler.fit_transform(train.loc[train_mask, sensor_cols])

            test_mask = test["regime"] == regime
            if test_mask.any():
                test.loc[test_mask, sensor_cols] = scaler.transform(test.loc[test_mask, sensor_cols])
            scalers[regime] = scaler
        print(f"[scale] fit {len(scalers)} per-regime scalers")
        return train, test, scalers
    else:
        scaler = MinMaxScaler()
        train[sensor_cols] = scaler.fit_transform(train[sensor_cols])
        test[sensor_cols] = scaler.transform(test[sensor_cols])
        print("[scale] fit a single global scaler")
        return train, test, scaler


# ---------------------------------------------------------------------------
# Step 8: Split by engine unit (never split by row - avoids data leakage)
# ---------------------------------------------------------------------------

def split_by_unit(train: pd.DataFrame, val_split: float = VAL_SPLIT):
    units = train["unit"].unique()
    train_units, val_units = train_test_split(
        units, test_size=val_split, random_state=RANDOM_STATE
    )

    train_split = train[train["unit"].isin(train_units)].copy()
    val_split_df = train[train["unit"].isin(val_units)].copy()

    print(f"[split] train engines={len(train_units)}, val engines={len(val_units)}")
    return train_split, val_split_df


# ---------------------------------------------------------------------------
# Step 9: Build sliding windows for sequence models (LSTM / Transformer)
# ---------------------------------------------------------------------------

def make_windows(df: pd.DataFrame, feature_cols: list, seq_len: int = SEQ_LEN):
    X, y, units_out = [], [], []
    skipped = 0

    for unit in df["unit"].unique():
        unit_df = df[df["unit"] == unit].sort_values("cycle")
        data = unit_df[feature_cols].values
        rul = unit_df["RUL"].values

        if len(data) < seq_len:
            skipped += 1
            continue

        for i in range(len(data) - seq_len + 1):
            X.append(data[i:i + seq_len])
            y.append(rul[i + seq_len - 1])
            units_out.append(unit)

    if skipped:
        print(f"[windows] skipped {skipped} engines shorter than seq_len={seq_len}")

    return np.array(X), np.array(y), np.array(units_out)


def make_test_windows(df: pd.DataFrame, feature_cols: list, seq_len: int = SEQ_LEN):
    """Test engines are evaluated only at their final cycle (matches RUL_FD00X.txt)."""
    X, units_out = [], []
    skipped = 0

    for unit in df["unit"].unique():
        unit_df = df[df["unit"] == unit].sort_values("cycle")
        data = unit_df[feature_cols].values

        if len(data) < seq_len:
            skipped += 1
            continue

        X.append(data[-seq_len:])
        units_out.append(unit)

    if skipped:
        print(f"[windows] test: skipped {skipped} engines shorter than seq_len={seq_len}")

    return np.array(X), np.array(units_out)


# ---------------------------------------------------------------------------
# Step 10: Save everything
# ---------------------------------------------------------------------------

def save_outputs(out_dir, subset, train_split, val_split_df, test, rul_test,
                  X_train, y_train, X_val, y_val, X_test, test_units):
    os.makedirs(out_dir, exist_ok=True)

    train_split.to_parquet(os.path.join(out_dir, f"train_{subset}.parquet"), index=False)
    val_split_df.to_parquet(os.path.join(out_dir, f"val_{subset}.parquet"), index=False)
    test.to_parquet(os.path.join(out_dir, f"test_{subset}.parquet"), index=False)
    rul_test.to_csv(os.path.join(out_dir, f"rul_test_{subset}.csv"), index=False)

    np.save(os.path.join(out_dir, f"X_train_{subset}.npy"), X_train)
    np.save(os.path.join(out_dir, f"y_train_{subset}.npy"), y_train)
    np.save(os.path.join(out_dir, f"X_val_{subset}.npy"), X_val)
    np.save(os.path.join(out_dir, f"y_val_{subset}.npy"), y_val)
    np.save(os.path.join(out_dir, f"X_test_{subset}.npy"), X_test)
    np.save(os.path.join(out_dir, f"test_units_{subset}.npy"), test_units)

    print(f"[save] all outputs written to {out_dir}/")


# ---------------------------------------------------------------------------
# Main pipeline
# ---------------------------------------------------------------------------

def run_pipeline(data_dir: str, out_dir: str, subset: str, seq_len: int = SEQ_LEN):
    print(f"\n{'=' * 60}\nProcessing {subset}\n{'=' * 60}")

    # 1-2: load
    train, test, rul_test = load_raw(data_dir, subset)

    # 3: label training RUL
    train = add_train_rul(train)

    # 4: cap RUL
    train = cap_rul(train)

    # 5: drop constant sensors
    train, test, dropped = drop_constant_sensors(train, test)

    # 6: cluster operating regimes (no-op for single-condition subsets)
    train, test, _ = add_regime_clusters(train, test, subset)

    # 7: normalize
    train, test, _ = normalize_sensors(train, test, subset)

    # 8: split by unit
    train_split, val_split_df = split_by_unit(train)

    # 9: sliding windows
    feature_cols = [c for c in train.columns if c.startswith("sensor_") or "op_setting" in c]
    X_train, y_train, _ = make_windows(train_split, feature_cols, seq_len)
    X_val, y_val, _ = make_windows(val_split_df, feature_cols, seq_len)
    X_test, test_units = make_test_windows(test, feature_cols, seq_len)

    print(f"[shapes] X_train={X_train.shape}, X_val={X_val.shape}, X_test={X_test.shape}")

    # 10: save
    save_outputs(out_dir, subset, train_split, val_split_df, test, rul_test,
                 X_train, y_train, X_val, y_val, X_test, test_units)

    return {
        "train": train_split, "val": val_split_df, "test": test, "rul_test": rul_test,
        "X_train": X_train, "y_train": y_train,
        "X_val": X_val, "y_val": y_val,
        "X_test": X_test, "test_units": test_units,
    }


Running it

In [ ]:
OUT_DIR = '/content/drive/MyDrive/AeroShield_AI/processed'  # or a local /content/ path

results = run_pipeline(
    data_dir=RAW_DIR,
    out_dir=OUT_DIR,
    subset='FD001',
    seq_len=30
)

# quick sanity check
print(results['X_train'].shape, results['y_train'].shape)
results['train'].head()


Processing FD001
[load] FD001: train=(20631, 26), test=(13096, 26), engines(train)=100, engines(test)=100
[filter] dropping constant sensors: ['sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']
[scale] fit a single global scaler
[split] train engines=80, val engines=20
[shapes] X_train=(14241, 30, 18), X_val=(3490, 30, 18), X_test=(100, 30, 18)
[save] all outputs written to /content/drive/MyDrive/AeroShield_AI/processed/
(14241, 30, 18) (14241,)


,unit,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_2,sensor_3,sensor_4,sensor_6,sensor_7,...,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21,RUL,regime
192,2,1,-0.0018,0.0006,100.0,0.204819,0.279049,0.152431,0.0,0.753623,...,0.047619,0.776119,0.264706,0.194963,0.252405,0.250000,0.620155,0.779205,125,0
193,2,2,0.0043,-0.0003,100.0,0.183735,0.349030,0.183660,1.0,0.792271,...,0.232143,0.855011,0.147059,0.160749,0.353213,0.333333,0.713178,0.710163,125,0
194,2,3,0.0018,0.0003,100.0,0.102410,0.376717,0.282073,0.0,0.851852,...,0.220238,0.829424,0.161765,0.209722,0.212774,0.250000,0.751938,0.732947,125,0
195,2,4,0.0035,-0.0004,100.0,0.141566,0.285808,0.233457,1.0,0.708535,...,0.148810,0.810235,0.073529,0.209000,0.295883,0.250000,0.767442,0.840238,125,0
196,2,5,0.0005,0.0004,100.0,0.156627,0.174188,0.342167,0.0,0.848631,...,0.238095,0.763326,0.088235,0.189545,0.237784,0.166667,0.806202,0.730737,125,0


Miscellaneous Subsets

In [ ]:
for subset in ['FD001', 'FD002', 'FD003', 'FD004']:
    run_pipeline(RAW_DIR, OUT_DIR, subset)


Processing FD001
[load] FD001: train=(20631, 26), test=(13096, 26), engines(train)=100, engines(test)=100
[filter] dropping constant sensors: ['sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']
[scale] fit a single global scaler
[split] train engines=80, val engines=20
[shapes] X_train=(14241, 30, 18), X_val=(3490, 30, 18), X_test=(100, 30, 18)
[save] all outputs written to /content/drive/MyDrive/AeroShield_AI/processed/

Processing FD002
[load] FD002: train=(53759, 26), test=(33991, 26), engines(train)=260, engines(test)=259
[filter] dropping constant sensors: none
[regime] FD002: clustered into 6 operating regimes
[scale] fit 6 per-regime scalers
[split] train engines=208, val engines=52
[windows] test: skipped 6 engines shorter than seq_len=30
[shapes] X_train=(37432, 30, 24), X_val=(8787, 30, 24), X_test=(253, 30, 24)
[save] all outputs written to /content/drive/MyDrive/AeroShield_AI/processed/

Processing FD003
[load] FD003: train=(24720, 26), test=(16596

In [ ]:
import shutil

# Create a zip archive of the processed data
shutil.make_archive('/content/processed_cmapss', 'zip', OUT_DIR)
print(f"Processed data zipped to /content/processed_cmapss.zip")

Processed data zipped to /content/processed_cmapss.zip
